# Publication-Quality Visualizations

Notebook ini menghasilkan figure PDF/SVG untuk laporan model traffic prediction. Karena beberapa plot memakai prediksi checkpoint LSTM, workflow disimpan sebagai `.ipynb`.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = Path("C:/AIproject/AWAI")

sys.path.insert(0, str(PROJECT_ROOT / "src"))
PROJECT_ROOT

WindowsPath('C:/AIproject/AWAI')

In [2]:
from datetime import datetime
import json

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

from traffic_prediction.config.settings import load_config
from traffic_prediction.data.processor import DataProcessor
from traffic_prediction.evaluation.baselines import (
    HistoricalAverageBaseline,
    LinearRegressionBaseline,
    PersistenceBaseline,
)
from traffic_prediction.features.offline import FeatureEngineer
from traffic_prediction.features.spatial import build_neighbor_mapping
from traffic_prediction.models.lstm import LSTMModelConfig, TrafficLSTM
from traffic_prediction.models.registry import ModelRegistry
from traffic_prediction.pipelines.offline import select_feature_columns

config = load_config(project_root=PROJECT_ROOT)
run_stamp = datetime.now().strftime("publication-figures-%Y%m%d-%H%M%S")
figure_dir = config.paths.figures_dir / run_stamp
figure_dir.mkdir(parents=True, exist_ok=True)

registry = ModelRegistry(config.paths.models_dir / "registry.json")
active_model = registry.get_active()
if active_model is None or active_model.model_type != "lstm":
    raise RuntimeError("Active model must be a trained LSTM before running this notebook")

model_dir = Path(active_model.artifact_path)
model_config_payload = json.loads((model_dir / "model_config.json").read_text(encoding="utf-8"))
training_history = pd.DataFrame(json.loads((model_dir / "training_history.json").read_text(encoding="utf-8"))["history"])

baseline_comparison = pd.read_csv(PROJECT_ROOT / "artifacts/reports/baseline-comparison-20260518-040123/baseline_comparison.csv")
horizon_metrics = pd.read_csv(PROJECT_ROOT / "artifacts/reports/horizon-diagnostics-20260518-044814/horizon_metrics.csv")
horizon_gaps = pd.read_csv(PROJECT_ROOT / "artifacts/reports/horizon-diagnostics-20260518-044814/lstm_vs_best_baseline_by_horizon.csv")

plt.rcParams.update({
    "figure.dpi": 140,
    "savefig.dpi": 300,
    "font.size": 10,
    "axes.titlesize": 12,
    "axes.labelsize": 10,
    "legend.fontsize": 9,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

figure_paths = []

def save_figure(fig, name: str) -> None:
    pdf_path = figure_dir / f"{name}.pdf"
    svg_path = figure_dir / f"{name}.svg"
    fig.tight_layout()
    fig.savefig(pdf_path, bbox_inches="tight")
    fig.savefig(svg_path, bbox_inches="tight")
    figure_paths.extend([str(pdf_path), str(svg_path)])
    plt.close(fig)

print("Figure run:", run_stamp)
print("Figure dir:", figure_dir)
print("Active model:", active_model.model_version)

Figure run: publication-figures-20260518-051036
Figure dir: C:\AIproject\AWAI\artifacts\figures\publication-figures-20260518-051036
Active model: lstm-real-20260518-032721


In [3]:
processor = DataProcessor(config.data, config.features)
traffic_raw, validation_report = processor.load_traffic_csv(config.paths.traffic_csv)
roads = processor.load_roads_csv(config.paths.roads_csv)
cleaned = processor.clean(traffic_raw)

neighbor_mapping = build_neighbor_mapping(roads, neighbor_count=config.features.spatial_neighbor_count)
feature_engineer = FeatureEngineer(
    neighbor_mapping=neighbor_mapping,
    lag_steps=config.features.lag_steps,
    rolling_windows=config.features.rolling_windows,
)
featured = feature_engineer.extract_features(cleaned)
feature_columns = select_feature_columns(featured)

train, validation, test, split_stats = processor.chronological_split(featured)
train_scaled = processor.fit_transform_train(train, feature_columns)
test_scaled = processor.transform_eval(test)

X_train, y_train, train_metadata = processor.create_sequences(train_scaled, feature_columns)
X_test, y_test, test_metadata = processor.create_sequences(test_scaled, feature_columns)

model_config = LSTMModelConfig(**model_config_payload["model_config"])
checkpoint = torch.load(model_config_payload["checkpoint_path"], map_location="cpu")
model = TrafficLSTM(model_config)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

def predict_in_batches(X: np.ndarray, batch_size: int = 2048) -> np.ndarray:
    outputs = []
    with torch.no_grad():
        for start in range(0, len(X), batch_size):
            batch = torch.as_tensor(X[start : start + batch_size], dtype=torch.float32)
            outputs.append(model(batch).numpy())
    return np.concatenate(outputs, axis=0)

def inverse_speed(values: np.ndarray) -> np.ndarray:
    return processor.scaler_store.inverse_transform_speed(values.reshape(-1, 1)).reshape(values.shape)

lstm_predictions_kmh = inverse_speed(predict_in_batches(X_test))
y_test_kmh = inverse_speed(y_test)

frequency = pd.Timedelta(config.data.frequency)
current_speed_index = feature_columns.index("current_speed")
baseline_feature_columns = [f"feature__{column}" for column in feature_columns]

def build_horizon_frame(X: np.ndarray, y: np.ndarray, metadata) -> pd.DataFrame:
    y_kmh = inverse_speed(y)
    last_step_features = X[:, -1, :]
    lag_1_kmh = inverse_speed(X[:, -1, current_speed_index])
    rows = []
    for sequence_index, item in enumerate(metadata):
        for horizon_index in range(y.shape[1]):
            row = {
                "sequence_index": sequence_index,
                "horizon_index": horizon_index,
                "road_id": item.road_id,
                "timestamp": pd.Timestamp(item.target_start) + horizon_index * frequency,
                "horizon_minutes": int((horizon_index + 1) * frequency.total_seconds() / 60),
                "actual_speed": float(y_kmh[sequence_index, horizon_index, 0]),
                "lag_1": float(lag_1_kmh[sequence_index]),
            }
            for column, value in zip(baseline_feature_columns, last_step_features[sequence_index]):
                row[column] = float(value)
            rows.append(row)
    return pd.DataFrame(rows)

train_eval = build_horizon_frame(X_train, y_train, train_metadata)
test_eval = build_horizon_frame(X_test, y_test, test_metadata)
test_eval["lstm"] = lstm_predictions_kmh.reshape(-1)

baselines = {
    "naive_persistence": PersistenceBaseline().fit(train_eval),
    "historical_average": HistoricalAverageBaseline().fit(train_eval),
    "linear_regression": LinearRegressionBaseline(feature_columns=baseline_feature_columns).fit(train_eval),
}
for name, baseline in baselines.items():
    test_eval[name] = baseline.predict(test_eval)

for model_name in ["lstm", "linear_regression", "historical_average", "naive_persistence"]:
    test_eval[f"abs_error_{model_name}"] = (test_eval["actual_speed"] - test_eval[model_name]).abs()

print("Rows for visualization:", len(test_eval))

Rows for visualization: 88400


In [4]:
fig, ax = plt.subplots(figsize=(7.0, 4.2))
ax.plot(training_history["epoch"], training_history["train_loss"], label="Train loss", linewidth=2)
ax.plot(training_history["epoch"], training_history["validation_loss"], label="Validation loss", linewidth=2)
best_epoch = int(training_history.loc[training_history["validation_loss"].idxmin(), "epoch"])
ax.axvline(best_epoch, color="0.35", linestyle="--", linewidth=1, label=f"Best epoch {best_epoch}")
ax.set_title("LSTM Training and Validation Loss")
ax.set_xlabel("Epoch")
ax.set_ylabel("MSE loss")
ax.grid(alpha=0.25)
ax.legend(frameon=False)
save_figure(fig, "learning_curve_lstm")

ordered = baseline_comparison.sort_values("rmse")
labels = ordered["model"].str.replace("lstm-real-20260518-032721", "LSTM", regex=False)
x = np.arange(len(ordered))
width = 0.38
fig, ax = plt.subplots(figsize=(7.2, 4.2))
ax.bar(x - width / 2, ordered["mae"], width, label="MAE", color="#4C78A8")
ax.bar(x + width / 2, ordered["rmse"], width, label="RMSE", color="#F58518")
ax.set_title("Overall Test Error by Model")
ax.set_ylabel("Error (km/h)")
ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=20, ha="right")
ax.grid(axis="y", alpha=0.25)
ax.legend(frameon=False)
save_figure(fig, "baseline_comparison_mae_rmse")

fig, ax = plt.subplots(figsize=(7.2, 4.2))
for model_name, group in horizon_metrics.groupby("model"):
    label = "LSTM" if model_name == "lstm" else model_name.replace("_", " ").title()
    ax.plot(group["horizon_minutes"], group["rmse"], marker="o", linewidth=2, label=label)
ax.set_title("RMSE by Forecast Horizon")
ax.set_xlabel("Forecast horizon (minutes)")
ax.set_ylabel("RMSE (km/h)")
ax.set_xticks(sorted(horizon_metrics["horizon_minutes"].unique()))
ax.grid(alpha=0.25)
ax.legend(frameon=False)
save_figure(fig, "horizon_rmse_by_model")

fig, ax = plt.subplots(figsize=(7.0, 4.0))
ax.bar(horizon_gaps["horizon_minutes"].astype(str), horizon_gaps["rmse_gap"], color="#E45756")
ax.set_title("LSTM RMSE Gap vs Best Baseline")
ax.set_xlabel("Forecast horizon (minutes)")
ax.set_ylabel("RMSE gap (km/h)")
ax.grid(axis="y", alpha=0.25)
save_figure(fig, "lstm_gap_vs_best_baseline")

In [5]:
sample = test_eval.sample(n=min(5000, len(test_eval)), random_state=42)
fig, ax = plt.subplots(figsize=(5.4, 5.0))
ax.scatter(sample["actual_speed"], sample["lstm"], s=8, alpha=0.25, color="#4C78A8", edgecolors="none")
limit_min = min(sample["actual_speed"].min(), sample["lstm"].min())
limit_max = max(sample["actual_speed"].max(), sample["lstm"].max())
ax.plot([limit_min, limit_max], [limit_min, limit_max], color="0.2", linestyle="--", linewidth=1)
ax.set_title("LSTM Actual vs Predicted Speed")
ax.set_xlabel("Actual speed (km/h)")
ax.set_ylabel("Predicted speed (km/h)")
ax.grid(alpha=0.25)
save_figure(fig, "actual_vs_predicted_lstm")

error_columns = ["abs_error_linear_regression", "abs_error_historical_average", "abs_error_naive_persistence", "abs_error_lstm"]
error_labels = ["Linear regression", "Historical average", "Naive persistence", "LSTM"]
fig, ax = plt.subplots(figsize=(7.2, 4.2))
ax.boxplot(
    [test_eval[column].to_numpy(dtype=float) for column in error_columns],
    labels=error_labels,
    showfliers=False,
    patch_artist=True,
    boxprops={"facecolor": "#D9E8F5", "color": "#4C78A8"},
    medianprops={"color": "#E45756", "linewidth": 1.5},
)
ax.set_title("Absolute Error Distribution")
ax.set_ylabel("Absolute error (km/h)")
ax.tick_params(axis="x", rotation=15)
ax.grid(axis="y", alpha=0.25)
save_figure(fig, "error_distribution_boxplot")

C:\Users\bagas\AppData\Local\Temp\ipykernel_33164\3228819923.py:16: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  ax.boxplot(


In [6]:
test_eval["hour"] = pd.to_datetime(test_eval["timestamp"]).dt.hour
test_eval["day_of_week"] = pd.to_datetime(test_eval["timestamp"]).dt.dayofweek
heatmap = test_eval.pivot_table(
    index="day_of_week",
    columns="hour",
    values="abs_error_lstm",
    aggfunc="mean",
).reindex(index=range(7), columns=range(24))

fig, ax = plt.subplots(figsize=(9.0, 3.8))
image = ax.imshow(heatmap.to_numpy(dtype=float), aspect="auto", cmap="viridis")
ax.set_title("LSTM Mean Absolute Error by Day and Hour")
ax.set_xlabel("Hour of day")
ax.set_ylabel("Day of week")
ax.set_xticks(range(0, 24, 2))
ax.set_yticks(range(7))
ax.set_yticklabels(["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"])
cbar = fig.colorbar(image, ax=ax)
cbar.set_label("MAE (km/h)")
save_figure(fig, "temporal_error_heatmap_lstm")

road_errors = test_eval.groupby("road_id", as_index=False)["abs_error_lstm"].mean()
spatial = roads.merge(road_errors, on="road_id", how="inner")
fig, ax = plt.subplots(figsize=(6.4, 5.4))
scatter = ax.scatter(
    spatial["mid_lon"],
    spatial["mid_lat"],
    c=spatial["abs_error_lstm"],
    cmap="magma",
    s=52,
    edgecolor="white",
    linewidth=0.5,
)
ax.set_title("Spatial Distribution of LSTM Mean Absolute Error")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.grid(alpha=0.20)
cbar = fig.colorbar(scatter, ax=ax)
cbar.set_label("Mean absolute error (km/h)")
save_figure(fig, "spatial_error_map_lstm")

In [7]:
manifest = {
    "run_id": run_stamp,
    "active_model_version": active_model.model_version,
    "created_at": datetime.now().isoformat(),
    "figure_dir": str(figure_dir),
    "figure_count": len(figure_paths),
    "figures": figure_paths,
    "notes": [
        "Each figure is saved as PDF and SVG.",
        "Prediction-based figures use the active LSTM checkpoint and the same test split as prior notebooks."
    ],
}
manifest_path = figure_dir / "visualization_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2, default=str), encoding="utf-8")
manifest

{'run_id': 'publication-figures-20260518-051036',
 'active_model_version': 'lstm-real-20260518-032721',
 'created_at': '2026-05-18T05:12:38.155165',
 'figure_dir': 'C:\\AIproject\\AWAI\\artifacts\\figures\\publication-figures-20260518-051036',
 'figure_count': 16,
 'figures': ['C:\\AIproject\\AWAI\\artifacts\\figures\\publication-figures-20260518-051036\\learning_curve_lstm.pdf',
  'C:\\AIproject\\AWAI\\artifacts\\figures\\publication-figures-20260518-051036\\learning_curve_lstm.svg',
  'C:\\AIproject\\AWAI\\artifacts\\figures\\publication-figures-20260518-051036\\baseline_comparison_mae_rmse.pdf',
  'C:\\AIproject\\AWAI\\artifacts\\figures\\publication-figures-20260518-051036\\baseline_comparison_mae_rmse.svg',
  'C:\\AIproject\\AWAI\\artifacts\\figures\\publication-figures-20260518-051036\\horizon_rmse_by_model.pdf',
  'C:\\AIproject\\AWAI\\artifacts\\figures\\publication-figures-20260518-051036\\horizon_rmse_by_model.svg',
  'C:\\AIproject\\AWAI\\artifacts\\figures\\publication-figu